CIAC_summer_of_tech
AI

In [ ]:

import pandas as pd
import numpy as np
from textblob import TextBlob
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib
import re

#loading data
df = pd.read_csv('CIAC_Data.csv')
df['datetime'] = pd.to_datetime(df['date'])

#features

df['word_count']      = df['content'].apply(lambda x: len(str(x).split()))
df['char_count']      = df['content'].apply(lambda x: len(str(x)))
df['sentiment']       = df['content'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df['num_hashtags']    = df['content'].apply(lambda x: str(x).count('#'))
df['num_mentions']    = df['content'].apply(lambda x: str(x).count('@'))
df['avg_word_length'] = df['char_count'] / (df['word_count'] + 1)


def count_emojis(text): #emoji count
    emoji_pattern = re.compile(
        "[\U0001F600-\U0001F64F"   # emoticons
        "\U0001F300-\U0001F5FF"   # symbols & pictographs
        "\U0001F680-\U0001F6FF"    # transport & map
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251]+", flags=re.UNICODE)
    return len(emoji_pattern.findall(str(text)))

df['emoji_count'] = df['content'].apply(count_emojis)

df['has_media'] = df['media'].apply(lambda x: 0 if str(x) in ['nan', '', '[]', 'None'] else 1)

df['hour']        = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek


company_post_count = df.groupby('inferred company')['id'].transform('count')
df['company_post_count'] = company_post_count


df = df[df['likes'] < df['likes'].quantile(0.99)].copy()

#train-test split
base_features = [
    'word_count', 'char_count', 'sentiment',
    'num_hashtags', 'num_mentions', 'avg_word_length',
    'has_media', 'hour', 'day_of_week',
    'emoji_count', 'company_post_count'
]

y = np.log1p(df['likes'])
X = df[base_features + ['inferred company']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#Target encode company
train_company_avg = (
    X_train.copy()
    .assign(likes=np.expm1(y_train))
    .groupby('inferred company')['likes']
    .mean()
)
global_avg = np.expm1(y_train).mean()

X_train = X_train.copy()
X_test  = X_test.copy()

X_train['company_target_enc'] = X_train['inferred company'].map(train_company_avg).fillna(global_avg)
X_test['company_target_enc']  = X_test['inferred company'].map(train_company_avg).fillna(global_avg)

X_train = X_train.drop(columns=['inferred company'])
X_test  = X_test.drop(columns=['inferred company'])

final_features = base_features + ['company_target_enc']

#model
print("Running hyperparameter search (this may take a few minutes)...")

param_dist = {
    'n_estimators':     [300, 500, 700],
    'learning_rate':    [0.01, 0.05, 0.1],
    'max_depth':        [4, 5, 6, 7],
    'subsample':        [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [3, 5, 7],
    'reg_alpha':        [0, 0.1, 0.5],
    'reg_lambda':       [0.5, 1.0, 2.0],
}

base_model = XGBRegressor(random_state=42, verbosity=0)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=30,          
    scoring='r2',
    cv=3,               
    random_state=42,
    n_jobs=-1,          
    verbose=1
)

search.fit(X_train[final_features], y_train)

print("\nBest Parameters Found:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

model = XGBRegressor(**search.best_params_, random_state=42, verbosity=0)
model.fit(
    X_train[final_features], y_train,
    eval_set=[(X_test[final_features], y_test)],
    verbose=False
)

#cv score
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train[final_features], y_train, cv=kf, scoring='r2')
print(f"\nCross-Validation R² Scores: {cv_scores.round(4)}")
print(f"Mean CV R²: {cv_scores.mean():.4f}  (+/- {cv_scores.std():.4f})")

#stats
preds_log      = model.predict(X_test[final_features])
rmse_log       = np.sqrt(mean_squared_error(y_test, preds_log))
r2             = r2_score(y_test, preds_log)
preds_original = np.expm1(preds_log)
y_test_orig    = np.expm1(y_test)
rmse_original  = np.sqrt(mean_squared_error(y_test_orig, preds_original))

print("\n" + "=" * 40)
print(f"RMSE (log scale):      {rmse_log:.4f}")
print(f"RMSE (original scale): {rmse_original:.2f}")
print(f"R2 Score:              {r2:.4f}")
print("=" * 40)

print("\nFeature Importance:")
importance = pd.Series(model.feature_importances_, index=final_features)
print(importance.sort_values(ascending=False).round(4))

#save
joblib.dump(model,             'like_predictor.pkl')
joblib.dump(train_company_avg, 'company_encoder.pkl')
joblib.dump(global_avg,        'global_avg.pkl')

print("\nModel saved as like_predictor.pkl")
print("Company encoder saved as company_encoder.pkl")


Running hyperparameter search (this may take a few minutes)...
Fitting 3 folds for each of 30 candidates, totalling 90 fits

Best Parameters Found:
  subsample: 0.9
  reg_lambda: 1.0
  reg_alpha: 0.5
  n_estimators: 500
  min_child_weight: 5
  max_depth: 7
  learning_rate: 0.01
  colsample_bytree: 0.7

Cross-Validation R² Scores: [0.794  0.7878 0.794  0.7966 0.8027]
Mean CV R²: 0.7950  (+/- 0.0048)

RMSE (log scale):      1.1607
RMSE (original scale): 1103.00
R2 Score:              0.7997

Feature Importance:
company_target_enc    0.5687
company_post_count    0.2514
num_hashtags          0.0387
emoji_count           0.0307
word_count            0.0300
char_count            0.0216
avg_word_length       0.0195
hour                  0.0121
sentiment             0.0105
num_mentions          0.0087
day_of_week           0.0081
has_media             0.0000
dtype: float32

Model saved as like_predictor.pkl
Company encoder saved as company_encoder.pkl
